# Random forest solubility model: contents of `protein_solubility_RF_backend.pkl`

A `.pkl` file stores a **trained model**, not the code that trained it, so it cannot be turned into a notebook directly. This notebook does what can be done from it:

1. Loads the file safely and shows everything stored inside it.
2. Shows the model settings, features and feature importances.
3. Provides a `predict_solubility()` function that gives a 0.0 to 1.0 score for one protein.
4. **Reconstructs** the training from the settings stored in the file, so you can reproduce and check the model on your own data. This part is a reconstruction. The original training code and the original train/test split are not in the file.

Keep `protein_solubility_RF_backend.pkl` in the same folder as this notebook. Parts 6 and 7 also need `protein_solubility_master_cleaned.csv` and your UniProt TSV with the Sequence column.

## 1. Setup

In [ ]:
# pip install pandas numpy scikit-learn matplotlib biopython joblib   (run once if a package is missing)
import re, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib.numpy_pickle import NumpyUnpickler

PKL = "protein_solubility_RF_backend.pkl"
MASTER_CSV = "protein_solubility_master_cleaned.csv"
UNIPROT_TSV = "uniprotkb_proteome_UP000000625_2026_10_08__1_.tsv"
AA = "ACDEFGHIKLMNPQRSTVWY"

## 2. Load the pickle safely
Pickle files can run arbitrary code when opened, so only open ones you trust. This loader refuses everything except scikit-learn, NumPy and joblib array classes, which is all this file needs. If it ever raises "blocked", the file contains something unexpected and you should not open it.

The file was saved with scikit-learn 1.6.1. A different installed version can print a version warning. The model should still load, but retrain it (part 7) if you want to be certain.

In [ ]:
ALLOWED_PREFIXES = ("sklearn.", "numpy", "joblib.numpy_pickle")

class SafeUnpickler(NumpyUnpickler):
    def find_class(self, module, name):
        if not module.startswith(ALLOWED_PREFIXES):
            raise RuntimeError(f"blocked {module}.{name}")
        return super().find_class(module, name)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")           # hides the scikit-learn version warning
    with open(PKL, "rb") as fh:
        bundle = SafeUnpickler(PKL, fh, True).load()

print("Keys stored in the file:", list(bundle.keys()))
rf, imputer = bundle["model"], bundle["imputer"]
feature_cols, threshold = bundle["feature_cols"], bundle["threshold"]

## 3. What is stored in the file

In [ ]:
print("Name:     ", bundle["model_name"])
print("Version:  ", bundle["model_version"])
print("Classes:  ", bundle["classes"])
print("Threshold:", threshold)
print("Features: ", bundle["n_features"])
print()
print("Metrics saved with the model (the file does not say which data they were measured on):")
pd.Series(bundle["metrics"]).round(4).to_frame("value")

In [ ]:
params = pd.Series(rf.get_params()).to_frame("value")
print(f"{len(rf.estimators_)} trees, deepest tree has depth {max(e.tree_.max_depth for e in rf.estimators_)}")
params

In [ ]:
# The 27 features, in the exact order the model expects, with the median used to fill missing values
pd.DataFrame({"feature": feature_cols, "fill_value_if_missing": imputer.statistics_.round(2)})

## 4. Feature importance

In [ ]:
imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
top = imp.head(15).iloc[::-1]
plt.figure(figsize=(7, 5))
plt.barh(top.index, top.values)
plt.title("Random forest: top 15 feature importances")
plt.xlabel("Mean decrease in impurity")
plt.tight_layout(); plt.show()
imp.round(4).to_frame("importance").head(15)

## 5. Predict one protein
`predict_solubility` takes a dictionary of feature values. Missing values are filled with the stored medians (applied directly, so it does not depend on the scikit-learn version). It returns the probability of being soluble (0.0 = not soluble, 1.0 = fully soluble) and the label at the stored threshold.

Percent features are on the 0 to 100 scale (for example `E_pct = 6.1`), and `MW_kDa` is in kilodaltons.

In [ ]:
def predict_solubility(features: dict):
    unknown = set(features) - set(feature_cols)
    if unknown:
        raise ValueError(f"Unknown feature names: {sorted(unknown)}")
    row = pd.DataFrame([{c: features.get(c, np.nan) for c in feature_cols}])
    X = row.fillna(dict(zip(feature_cols, imputer.statistics_)))    # median fill, same as the saved imputer
    p = float(rf.predict_proba(X.values)[0, 1])      # the saved forest was fitted without column names
    return {"p_soluble": round(p, 4),
            "label": bundle["classes"][int(p >= threshold)]}

# Example: only a few values given, the rest are filled with medians
predict_solubility({"MW_kDa": 30.0, "pI": 5.5, "GRAVY": -0.3, "E_pct": 8.0, "K_pct": 6.0})

## 6. Compute the same features from sequences
This rebuilds the 27 features from the amino acid sequences so the model can be scored and retrained. `MW_kDa` and `pI` come from the master table.

Note: the `Instability_Index` and `GRAVY` here come from Biopython. If the original model used a different implementation, small differences are possible.

In [ ]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

def seq_features(seq):
    seq = "".join(c for c in str(seq).upper() if c in AA)
    n = len(seq)
    if n < 10:
        return None
    pa = ProteinAnalysis(seq)
    pct = {a: 100 * seq.count(a) / n for a in AA}
    f = {f"{a}_pct": pct[a] for a in AA}
    f["GRAVY"] = pa.gravy()
    f["Instability_Index"] = pa.instability_index()
    f["Aliphatic_Index"] = pct["A"] + 2.9 * pct["V"] + 3.9 * (pct["I"] + pct["L"])
    f["Hydrophobic_pct"] = sum(pct[a] for a in "AVILMFWC")
    f["Charged_pct"] = sum(pct[a] for a in "DEKRH")
    return f

m = pd.read_csv(MASTER_CSV)
t = pd.read_csv(UNIPROT_TSV, sep="\t")
cols = {c.lower(): c for c in t.columns}
loc_col = next((c for l, c in cols.items() if "ordered locus" in l), None) or cols["gene names"]
pairs = []
for i, names in enumerate(t[loc_col].fillna("")):
    for tok in re.split(r"[;,\s]+", str(names)):
        if re.fullmatch(r"b\d{4}", tok) or re.fullmatch(r"JW\d{4}", tok):
            pairs.append((tok, t[cols["sequence"]].iloc[i]))
seqmap = dict(pd.DataFrame(pairs, columns=["k", "s"]).drop_duplicates("k").values)
m["Sequence"] = m["B number"].map(seqmap).fillna(m["JW_ID"].map(seqmap))

F = pd.DataFrame([f if f else {} for f in m["Sequence"].apply(lambda s: seq_features(s) if isinstance(s, str) else None)], index=m.index)
m = pd.concat([m, F], axis=1)
m["MW_kDa"], m["pI"] = m["Calculated MW (kDa)"], m["Calculated pI"]
m = m[m["GRAVY"].notna()].copy()

lab = m[m["has_solubility_label"] == True].drop_duplicates(["B number", "MW_kDa", "pI", "Solubility (%)"]).copy()
lab["y"] = (lab["Solubility (%)"] >= 50).astype(int)
print(f"Labeled proteins with sequences: {len(lab)} | Soluble share: {lab['y'].mean():.1%}")

## 7. Reconstruct the training and check the model on your data
The settings come straight from the pickle (300 trees, `class_weight="balanced"`, `random_state=42`, median imputation). The split is grouped by gene so the same gene never lands on both sides. Because the original split is unknown, these numbers will not match the saved metrics exactly.

**Do not score the saved model on this data and call it a test result.** It was almost certainly trained on many of these same proteins, so its accuracy here would be inflated. The retrained model below is scored only on proteins it has not seen.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

X, y, g = lab[feature_cols], lab["y"], lab["B number"]
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(X, y, g))
assert not set(g.iloc[tr]) & set(g.iloc[te])

rf_params = {k: v for k, v in rf.get_params().items()}          # exactly the settings stored in the pickle
pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                 ("rf", RandomForestClassifier(**rf_params))])
pipe.fit(X.iloc[tr], y.iloc[tr])
prob = pipe.predict_proba(X.iloc[te])[:, 1]
pred = (prob >= threshold).astype(int)

mine = {"accuracy": accuracy_score(y.iloc[te], pred), "precision": precision_score(y.iloc[te], pred),
        "recall": recall_score(y.iloc[te], pred), "f1": f1_score(y.iloc[te], pred),
        "roc_auc": roc_auc_score(y.iloc[te], prob)}
pd.DataFrame({"saved in pickle": bundle["metrics"], "retrained here (held-out test)": mine}).round(4)

In [ ]:
# A steadier estimate: 3 x repeated 5-fold cross-validation, grouped by gene
accs, aucs = [], []
for seed in range(3):
    rng = np.random.default_rng(seed)
    ug = g.unique()
    fold = dict(zip(rng.permutation(ug), np.arange(len(ug)) % 5))
    f = g.map(fold).values
    oof = np.zeros(len(X))
    for k in range(5):
        p = Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("rf", RandomForestClassifier(**{**rf_params, "random_state": seed}))])
        p.fit(X[f != k], y[f != k])
        oof[f == k] = p.predict_proba(X[f == k])[:, 1]
    accs.append(accuracy_score(y, oof >= threshold)); aucs.append(roc_auc_score(y, oof))
print(f"Grouped CV accuracy: {np.mean(accs):.1%}   ROC-AUC: {np.mean(aucs):.3f}")
print("Confusion matrix on the test split [[TN FP], [FN TP]]:\n", confusion_matrix(y.iloc[te], pred))

## 8. Save a model for later use
Saves the retrained pipeline, together with the feature order and threshold, so it can be loaded in a backend. Use `joblib.dump` to write it and load it only on a machine and scikit-learn version you control.

In [ ]:
import joblib
joblib.dump({"pipeline": pipe, "feature_cols": feature_cols, "threshold": threshold,
             "classes": bundle["classes"], "metrics_grouped_cv": {"accuracy": float(np.mean(accs)), "roc_auc": float(np.mean(aucs))}},
            "rf_retrained.joblib")
print("Saved rf_retrained.joblib")